# Day 20 Lab — Model Serving & Inference Optimization
### Cloud fallback (Google Colab / Kaggle)

**Use this only if your laptop cannot run the lab** — under 8 GB RAM, or setup failed
for a reason you cannot fix. The lab is designed for your own hardware, and the whole
point of §0–§3 is measuring *your* machine.

Grading is **not** affected. The rubric rewards the clarity of your measurements and
reasoning, never absolute speed. You must, however, **declare that you used the cloud
fallback in REFLECTION §1** — this notebook records it into `hardware.json` for you.

This notebook produces the **exact same artifact filenames** as the laptop path, so
`make verify` and the rubric work unchanged. At the end you download a zip and commit
its contents to your own repo.

---
**Runtime menu → Change runtime type:** CPU is fine and is the default here. A T4 GPU
is faster but needs the optional CUDA build cell (~8 min). Both are acceptable.

> Kaggle only: turn **Internet ON** in the notebook settings sidebar, or the model
> download will fail.


## 1. Configuration


In [ ]:
# Point this at YOUR fork, so the artifacts you commit come from your repo.
REPO_URL = "https://github.com/<your-username>/Day20-Track2-ModelServing-Lab.git"

# 'cpu'  = prebuilt CPU binary. Always works. Slower but sufficient for the whole lab.
# 'cuda' = compile llama.cpp with CUDA (~8 min). Only if you selected a GPU runtime.
#          llama.cpp publishes no prebuilt Linux CUDA binary, so this must be built.
RUNTIME = "cpu"

# Shorter load tests keep a free-tier session inside its time budget.
LOAD_DURATION = "1m"


## 2. Detect the environment


In [ ]:
import os, sys, subprocess, pathlib, shutil

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
ENV = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'unknown'
WORK = pathlib.Path('/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '.')
print('environment :', ENV)
print('workdir     :', WORK)

# This is what lands in hardware.json and satisfies rubric item 1.
os.environ['LAB_RUNTIME_ENV'] = ENV

print()
print(subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip(), 'vCPUs')
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'none (CPU runtime)')


## 3. Clone the lab and install dependencies

Four pure-Python packages. No compiler, and no `llama-cpp-python`.


In [ ]:
LAB = WORK / 'Day20-Track2-ModelServing-Lab'
if not LAB.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB)], check=True)
os.chdir(LAB)
print('cwd:', os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
               check=True)
print('deps installed')


## 4. Probe hardware + fetch the llama.cpp runtime


In [ ]:
!{sys.executable} labs/00-setup/detect-hardware.py

if RUNTIME == 'cpu':
    # Force the plain CPU asset. The auto-picker would choose the Vulkan build when it
    # sees an NVIDIA GPU, but Colab/Kaggle images normally ship no Vulkan driver, so
    # that build would have no device to bind to.
    sys.path.insert(0, 'lib')
    import labkit
    asset = f'llama-{labkit.LLAMA_CPP_BUILD}-bin-ubuntu-x64.tar.gz'
    print('fetching', asset)
    !{sys.executable} labs/00-setup/fetch-runtime.py --asset {asset}
else:
    print('RUNTIME=cuda -> skipping the prebuilt fetch; run the CUDA build cell below.')


### 4b. Optional: build with CUDA (GPU runtimes only, ~8 minutes)

Skip this cell entirely if `RUNTIME = 'cpu'`. Running it also earns bonus **B1** and
sets you up for challenge **C6** (Vulkan vs CUDA), since you can fetch the Vulkan
build afterwards and compare.


In [ ]:
if RUNTIME == 'cuda':
    # NOTE: we call cmake directly rather than `make build-llama`. The Makefile
    # targets require a .venv, and this notebook installs into the system Python.
    !apt-get -qq install -y cmake build-essential > /dev/null
    sys.path.insert(0, 'lib')
    import labkit
    BUILD = labkit.LLAMA_CPP_BUILD
    !test -d bonus/llama.cpp || git clone --depth 1 --branch {BUILD} \
        https://github.com/ggml-org/llama.cpp bonus/llama.cpp
    !cd bonus/llama.cpp && cmake -B build -DGGML_CUDA=ON -DGGML_NATIVE=ON \
        -DCMAKE_BUILD_TYPE=Release > /dev/null
    !cd bonus/llama.cpp && cmake --build build -j --config Release 2>&1 | tail -5
    !ls -la bonus/llama.cpp/build/bin/llama-server bonus/llama.cpp/build/bin/llama-bench
    print('CUDA build ready -- every lab script picks it up automatically.')
else:
    print('skipped (RUNTIME != cuda)')


## 5. Download Gemma 4 E2B (~5.2 GB)

Two quantizations of one model: `UD-Q4_K_XL` (primary) and `UD-Q2_K_XL` (comparison).
Apache-2.0 and ungated — no Hugging Face token needed.


In [ ]:
!{sys.executable} labs/00-setup/download-model.py
!cat models/active.json


## 6. Track 01 — Measure

TTFT, TPOT and percentiles for both quantizations, then the thread-count sweep.
**Screenshot the output of both cells** (screenshot 02, plus the optional tune shot).

On 2 vCPUs this is slow — expect several minutes per quantization. That is itself a
measurement worth commenting on in your reflection.


In [ ]:
!{sys.executable} labs/01-measure/benchmark.py


In [ ]:
# Cloud VMs give you few cores, so the thread curve is short but still real.
!{sys.executable} labs/01-measure/tune.py


## 7. Track 02 — Serve

The server runs in the background of this notebook, exactly as it would in a second
terminal on a laptop.


In [ ]:
import time, httpx, subprocess

srv = subprocess.Popen([sys.executable, 'labs/02-serve/serve.py'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for i in range(300):
    try:
        if httpx.get('http://127.0.0.1:8080/health', timeout=2).status_code == 200:
            print(f'server healthy after {i}s'); break
    except Exception:
        pass
    time.sleep(1)
else:
    print('server did not come up -- run labs/02-serve/serve.py in a cell to see why')


In [ ]:
# Rubric items 6 + 7 -- screenshot this output.
!{sys.executable} labs/02-serve/smoke-test.py


In [ ]:
# Rubric item 8 -- screenshot each summary table.
!{sys.executable} -m locust -f labs/02-serve/load-test.py --headless -u 10 -r 2 \
    -t {LOAD_DURATION} --host http://localhost:8080 \
    --csv benchmarks/locust-10 --only-summary


In [ ]:
# Run the 50-user load and sample /metrics at the same time (item 9 needs the overlap).
rec = subprocess.Popen([sys.executable, 'labs/02-serve/record-metrics.py',
                        '--duration', '60', '--label', 'u50'])
!{sys.executable} -m locust -f labs/02-serve/load-test.py --headless -u 50 -r 10 \
    -t {LOAD_DURATION} --host http://localhost:8080 \
    --csv benchmarks/locust-50 --only-summary
rec.wait()


In [ ]:
# Rubric item 10.
!{sys.executable} labs/02-serve/load-report.py


## 8. Track 03 — Integrate


In [ ]:
!{sys.executable} labs/03-integrate/pipeline.py


## 9. Verify, then take your artifacts home

`make verify` will still flag REFLECTION and the screenshots — you fill those in on
your own machine. Everything under `benchmarks/`, plus `hardware.json` and
`models/active.json`, is what you commit.


In [ ]:
srv.terminate()
!{sys.executable} scripts/verify.py; echo "(exit $?)"
print()
!ls -la benchmarks/ hardware.json models/active.json


In [ ]:
# Zip ONLY the evidence -- never the ~5 GB of weights.
ZIP = WORK / 'day20-artifacts.zip'
ZIP.unlink(missing_ok=True)
!cd {LAB} && zip -q -r {ZIP} benchmarks hardware.json models/active.json
!ls -lh {ZIP}
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))
else:
    print('Kaggle: find it in the Output panel on the right.')


## 10. Finish on your own machine

1. Unzip into your local clone: `benchmarks/`, `hardware.json`, `models/active.json`.
2. Replace every **"required — replace this line"** section in `benchmarks/*.md`
   with your own observations. `make verify` fails while any remain.
3. Fill in `submission/REFLECTION.md`. **In §1, say you used the cloud fallback and
   why** — `hardware.json` already records `runtime_environment`.
4. Add your 5 screenshots (take them from this notebook's cell outputs).
5. `make verify` → exit 0, then push to your **public** repo and submit the URL.

### One caveat worth writing about

A cloud VM is not your laptop: different core count, different memory bandwidth, a
hypervisor between you and the silicon, and noisy neighbours. Your numbers are valid
for *this VM* and are not comparable to a classmate's laptop — which was already true
of the laptop path. Note in §5 that your tuning result describes the VM you were given.

If the session disconnects mid-run, re-run from section 3 — the clone and model
download are the only slow parts, and both skip work that is already on disk.
